# 04 - Checkpoint Submissions
Load ArcFace head checkpoints from `../checkpoints`, rebuild the backbone inference pipeline, and create two submission files per checkpoint: one plain cosine similarity submission and one k-reciprocal reranked submission.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
from src.models import ArcFaceHeadModel, build_backbone
from src.utils import get_device, set_seed

RANDOM_SEED = 42
set_seed(RANDOM_SEED)
device = get_device()
device

device(type='cuda')

## Config

In [2]:
config = {
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints"),
    "submission_dir": Path("../submissions/checkpoint_submissions"),
    "batch_size": 32,
    "num_workers": 2,
    "seed": RANDOM_SEED,
    "rerank": {
        "use_rerank": True,
        "k1": 20,
        "k2": 6,
        "lambda_value": 0.3,
    },
}

config["submission_dir"].mkdir(parents=True, exist_ok=True)
config

{'data_dir': PosixPath('../data'),
 'checkpoint_dir': PosixPath('../checkpoints'),
 'submission_dir': PosixPath('../submissions/checkpoint_submissions'),
 'batch_size': 32,
 'num_workers': 2,
 'seed': 42,
 'rerank': {'use_rerank': True, 'k1': 20, 'k2': 6, 'lambda_value': 0.3}}

## Helpers

In [3]:
def list_checkpoint_paths(checkpoint_dir: Path):
    return sorted(checkpoint_dir.glob("*.pth"))


def resolve_backbone_name(checkpoint_config: dict):
    backbone_name = checkpoint_config.get("backbone_name")
    if backbone_name is None:
        backbone_name = checkpoint_config.get("megadescriptor_model")
    if backbone_name is None:
        raise ValueError("Checkpoint config does not contain 'backbone_name' or 'megadescriptor_model'.")
    return backbone_name


def load_test_bundle(model, config: dict, input_size: int):
    pairs_df = data.load_test_pairs_df(config["data_dir"])
    unique_images = sorted(set(pairs_df["query_image"]) | set(pairs_df["gallery_image"]))
    test_df = pd.DataFrame({"filename": unique_images})

    test_loader = data.create_test_loader(
        model,
        test_df,
        img_dir=config["data_dir"] / "test" / "test",
        input_size=input_size,
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )
    return pairs_df, test_loader


def build_head_model_from_checkpoint(checkpoint: dict, backbone_out_dim: int):
    checkpoint_config = checkpoint.get("config", {})
    num_classes = int(checkpoint["num_classes"])

    model = ArcFaceHeadModel(
        input_dim=backbone_out_dim,
        num_classes=num_classes,
        embedding_dim=int(checkpoint_config.get("embedding_dim", 256)),
        hidden_dim=int(checkpoint_config.get("hidden_dim", 512)),
        margin=float(checkpoint_config.get("arcface_margin", 0.5)),
        scale=float(checkpoint_config.get("arcface_scale", 64.0)),
        dropout=float(checkpoint_config.get("dropout", 0.3)),
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    return model


def create_submissions_for_checkpoint(checkpoint_path: Path, config: dict):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    checkpoint_config = checkpoint.get("config", {})

    backbone_name = resolve_backbone_name(checkpoint_config)
    input_size = int(checkpoint_config["input_size"])

    print(f"Loading checkpoint: {checkpoint_path.name}")
    print(f"  backbone={backbone_name}")
    print(f"  input_size={input_size}")

    backbone = build_backbone(backbone_name, pretrained=True).to(device)
    backbone.eval()

    backbone_out_dim = getattr(backbone, "num_features", None)
    if backbone_out_dim is None:
        raise ValueError("Backbone output dimension not found; pass backbone_out_dim")

    head_model = build_head_model_from_checkpoint(checkpoint, backbone_out_dim).to(device)
    head_model.eval()

    pairs_df, test_loader = load_test_bundle(backbone, config, input_size)

    checkpoint_stem = checkpoint_path.stem
    plain_submission_path = config["submission_dir"] / f"{checkpoint_stem}_submission.csv"
    rerank_submission_path = config["submission_dir"] / f"{checkpoint_stem}_submission_rerank.csv"

    inference.create_submission_backbone(
        backbone,
        head_model,
        device,
        pairs_df,
        test_loader,
        output_path=plain_submission_path,
        use_rerank=False,
    )

    inference.create_submission_backbone(
        backbone,
        head_model,
        device,
        pairs_df,
        test_loader,
        output_path=rerank_submission_path,
        use_rerank=config["rerank"]["use_rerank"],
        k1=config["rerank"]["k1"],
        k2=config["rerank"]["k2"],
        lambda_value=config["rerank"]["lambda_value"],
    )

    return {
        "checkpoint": checkpoint_path.name,
        "plain_submission": plain_submission_path,
        "rerank_submission": rerank_submission_path,
    }


## Available Checkpoints

In [4]:
checkpoint_paths = list_checkpoint_paths(config["checkpoint_dir"])
if not checkpoint_paths:
    raise FileNotFoundError(f"No .pth checkpoints found in {config['checkpoint_dir']}")

checkpoint_paths

[PosixPath('../checkpoints/arcface_best.pth')]

## Build Submissions

In [5]:
results = []
for checkpoint_path in checkpoint_paths:
    results.append(create_submissions_for_checkpoint(checkpoint_path, config))

results_df = pd.DataFrame(results)
results_df

Loading checkpoint: arcface_best.pth
  backbone=hf-hub:BVRA/MegaDescriptor-L-384
  input_size=384


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

,checkpoint,plain_submission,rerank_submission
0,arcface_best.pth,../submissions/checkpoint_submissions/arcface_...,../submissions/checkpoint_submissions/arcface_...
